# Jani — segmentación de hoja y lesión (severidad)

Segundo modelo de Jani. Para cada foto separa **fondo / hoja / síntoma** píxel por píxel. Con eso la app calcula:

- **Severidad** = píxeles de síntoma ÷ píxeles de hoja (lo que la propuesta llama "segmentación en dos etapas").
- **¿Hay una hoja en la foto?** Si casi no hay píxeles de hoja, pide repetir la foto.
- **Dónde está el daño**: pinta las lesiones en rojo sobre la foto.

**Cómo usarlo (no hay que editar nada):**
1. En Colab: `Archivo → Subir notebook` y elige este archivo.
2. `Entorno de ejecución → Cambiar tipo de entorno → GPU T4 → Guardar`.
3. `Entorno de ejecución → Ejecutar todo`. Tarda unos **20–30 minutos**.
4. Al final se descarga **`jani_leafseg.zip`**. Descomprímelo dentro de `app/public/models/leafseg-v1/` del repositorio.

**Datos:** 500 hojas de café de Brasil con máscaras dibujadas a mano (BRACOL, repositorio de Esgario et al., licencia
MIT): 400 para entrenar, 50 para elegir el modelo y 50 para la prueba final. Para el umbral de "¿hay una hoja?" se usan
además 300 hojas de JMuBEN (Kenia) y 300 fotos que no son hojas (Food-101).

In [ ]:
!pip -q install onnx onnxruntime onnxscript datasets

## Configuración
No hace falta cambiar nada.

In [ ]:
# Candidatos: LR-ASPP (cabeza de segmentación ligera de MobileNetV3) con dos tamaños de columna vertebral.
MODELS = ["lraspp_mobilenet_v3_small", "lraspp_mobilenet_v3_large"]
IMG_SIZE = 256          # la app estira la foto a 256×256, igual que aquí
EPOCHS = 60
BATCH = 16
LR = 1e-3
SEED = 42
CLASSES = ["fondo", "hoja", "sintoma"]          # orden de los canales de salida (la app depende de él)
MASK_COLORS = {(0, 0, 0): 0, (0, 176, 0): 1, (255, 0, 0): 2}   # colores de las máscaras de BRACOL

# Niveles de severidad de BRACOL (% del área de la hoja con síntomas), tal como los define el dataset.
SEVERITY_LEVELS = [("sana", 0.001), ("muy_baja", 0.05), ("baja", 0.10), ("alta", 0.15), ("muy_alta", 1.0)]

LEAF_GATE_SAMPLES = 300  # hojas de JMuBEN y fotos sin hoja (Food-101) para fijar el umbral de "¿hay una hoja?"
MAX_MODEL_MB = 10
INT8_MAX_MIOU_DIFF = 0.02   # int8 aceptable si su mIoU difiere <= 0,02 del fp32...
INT8_MAX_PIXEL_DISAGREE = 0.03  # ...y cambia la clase de <= 3 % de los píxeles

OUT_DIR = "jani_leafseg"
SMOKE_TEST = False       # True solo para probar el notebook sin GPU con imágenes sintéticas

In [ ]:
import os, io, json, math, random, shutil, time, zipfile, urllib.request, warnings
from collections import Counter
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
import torchvision
from torchvision.models import MobileNet_V3_Large_Weights, MobileNet_V3_Small_Weights
from torchvision.models.segmentation.lraspp import _lraspp_mobilenetv3
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)
if SMOKE_TEST:
    EPOCHS, BATCH, LEAF_GATE_SAMPLES = 1, 4, 0
    matplotlib.use("Agg"); plt.show = lambda *a, **k: None
else:
    assert device == "cuda", "Activa la GPU: Entorno de ejecución → Cambiar tipo de entorno → T4"
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet (columnas vertebrales preentrenadas)

## 1. Datos: máscaras de BRACOL

Se baja el repositorio público `esgario/lara2018` (MIT) y se usan sus carpetas `segmentation/dataset/images` y
`annotations` con la división original en `train` / `val` / `test`.

In [ ]:
UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}
REPO_ZIP = "https://codeload.github.com/esgario/lara2018/zip/refs/heads/master"

def download(url, path, retries=5):
    for k in range(retries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=600) as r, open(path + ".part", "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20)
            os.replace(path + ".part", path); return
        except Exception as e:
            if k == retries - 1: raise
            print("  reintentando descarga:", e); time.sleep(3 * 2 ** k)

def mask_to_index(m):
    """Máscara RGB de BRACOL -> índices 0/1/2. Píxeles de otro color van al color conocido más cercano."""
    a = np.asarray(m.convert("RGB"), dtype=np.int32)
    code = (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]
    out = np.full(code.shape, 255, np.uint8)
    for (r, g, b), k in MASK_COLORS.items(): out[code == ((r << 16) | (g << 8) | b)] = k
    odd = out == 255
    if odd.any():                                    # bordes con colores intermedios: el más cercano
        cols = np.array(list(MASK_COLORS), dtype=np.int32)
        d = ((a[odd][:, None, :] - cols[None]) ** 2).sum(-1)
        out[odd] = np.array(list(MASK_COLORS.values()), dtype=np.uint8)[d.argmin(-1)]
    return out

def load_split(root, split):
    items = []
    img_dir, ann_dir = f"{root}/images/{split}", f"{root}/annotations/{split}"
    for f in sorted(os.listdir(img_dir)):
        stem = os.path.splitext(f)[0]
        mp = f"{ann_dir}/{stem}_mask.png"
        if os.path.exists(mp): items.append((f"{img_dir}/{f}", mp))
    return items

def make_smoke_data(root):
    """Hojas sintéticas (elipse verde con manchas rojas) con su máscara, misma estructura que BRACOL."""
    rng = np.random.default_rng(0)
    for split, n in [("train", 12), ("val", 4), ("test", 4)]:
        os.makedirs(f"{root}/images/{split}", exist_ok=True); os.makedirs(f"{root}/annotations/{split}", exist_ok=True)
        for i in range(n):
            h, w = 160, 320
            yy, xx = np.mgrid[:h, :w]
            leaf = ((xx - w / 2) / (w * 0.42)) ** 2 + ((yy - h / 2) / (h * 0.40)) ** 2 <= 1
            spots = np.zeros_like(leaf)
            for _ in range(rng.integers(0, 6)):
                cy, cx, r = rng.integers(20, h - 20), rng.integers(40, w - 40), rng.integers(4, 14)
                spots |= (yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2
            spots &= leaf
            img = np.full((h, w, 3), 245, np.uint8)
            img[leaf] = (40, 140, 50); img[spots] = (150, 90, 30)
            img = np.clip(img + rng.normal(0, 8, img.shape), 0, 255).astype(np.uint8)
            mask = np.zeros((h, w, 3), np.uint8); mask[leaf] = (0, 176, 0); mask[spots] = (255, 0, 0)
            Image.fromarray(img).save(f"{root}/images/{split}/{i}.jpg")
            Image.fromarray(mask).save(f"{root}/annotations/{split}/{i}_mask.png")

if SMOKE_TEST:
    SEG_ROOT = "data/smoke_seg"; shutil.rmtree(SEG_ROOT, ignore_errors=True); make_smoke_data(SEG_ROOT)
else:
    SEG_ROOT = "data/lara2018-master/segmentation/dataset"
    if not os.path.isdir(SEG_ROOT):
        os.makedirs("data", exist_ok=True)
        print("Descargando el repositorio lara2018 (~120 MB)...")
        download(REPO_ZIP, "data/lara2018.zip")
        with zipfile.ZipFile("data/lara2018.zip") as z:
            z.extractall("data", members=[n for n in z.namelist() if "/segmentation/dataset/" in n])
        os.remove("data/lara2018.zip")

splits = {s: load_split(SEG_ROOT, s) for s in ("train", "val", "test")}
print({s: len(v) for s, v in splits.items()})
assert all(splits.values()), "Faltan imágenes o máscaras: revisa la descarga."

## 2. Preparación

Las fotos se **estiran** a 256×256 (igual que la app) y se guardan en memoria. La severidad real de cada hoja se calcula
con la máscara a resolución completa.

In [ ]:
def severity_of(idx):
    leaf = int((idx == 1).sum() + (idx == 2).sum())
    return float((idx == 2).sum()) / leaf if leaf else 0.0

def level_of(sev):
    for k, (name, upper) in enumerate(SEVERITY_LEVELS):
        if sev <= upper: return k
    return len(SEVERITY_LEVELS) - 1

def prepare(items, size):
    X, Y, S = [], [], []
    for ip, mp in items:
        with Image.open(ip) as im: x = np.asarray(im.convert("RGB").resize((size, size), Image.BILINEAR))
        with Image.open(mp) as m:
            full = mask_to_index(m)
            y = np.asarray(Image.fromarray(full).resize((size, size), Image.NEAREST))
        X.append(x); Y.append(y); S.append(severity_of(full))
    return (torch.tensor(np.stack(X)).permute(0, 3, 1, 2).contiguous(), torch.tensor(np.stack(Y)).long(), np.array(S))

CACHE = int(IMG_SIZE * 1.25)     # se guarda un poco más grande para recortes aleatorios al entrenar
data = {"train": prepare(splits["train"], CACHE)}
for s in ("val", "test"): data[s] = prepare(splits[s], IMG_SIZE)
for s, (X, Y, S) in data.items():
    px = np.bincount(Y.flatten().numpy(), minlength=3) / Y.numel()
    print(f"{s}: {len(X)} hojas | píxeles fondo/hoja/síntoma = {px[0]:.2f}/{px[1]:.2f}/{px[2]:.3f} | "
          f"severidad media {S.mean():.3f} | niveles {dict(Counter(SEVERITY_LEVELS[level_of(v)][0] for v in S))}")
pix = np.bincount(data["train"][1].flatten().numpy(), minlength=3).astype(float)
CLASS_WEIGHTS = torch.tensor((pix.sum() / (3 * np.maximum(pix, 1))) ** 0.5, dtype=torch.float32)
print("Pesos por clase (síntoma es escaso):", CLASS_WEIGHTS.numpy().round(2))

## 3. Entrenar, evaluar y exportar cada modelo

Aumentación conjunta de foto y máscara (recorte, giros de 90°, espejos, color). Pérdida: entropía cruzada con pesos +
Dice del síntoma. Se guarda la época con mejor IoU de síntoma en validación.

In [ ]:
def build(name):
    if name == "lraspp_mobilenet_v3_small":
        bb = torchvision.models.mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1, dilated=True)
    else:
        bb = torchvision.models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1, dilated=True)
    return _lraspp_mobilenetv3(bb, len(CLASSES))

class Export(nn.Module):
    """Envuelve el modelo para que el ONNX tenga una sola salida 'logits' [1, 3, H, W]."""
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(x)["out"]

mean_t = torch.tensor(MEAN).view(1, 3, 1, 1); std_t = torch.tensor(STD).view(1, 3, 1, 1)
def normalize(x_uint8): return (x_uint8.float() / 255 - mean_t.to(x_uint8.device)) / std_t.to(x_uint8.device)

def augment(x, y):
    """x uint8 [B,3,C,C], y long [B,C,C] -> recorte aleatorio de IMG_SIZE + giros/espejos + color."""
    B, _, C, _ = x.shape
    xs, ys = [], []
    for i in range(B):
        s = random.randint(int(C * 0.65), C); t, l = random.randint(0, C - s), random.randint(0, C - s)
        xi = F.interpolate(x[i:i+1, :, t:t+s, l:l+s].float(), size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        yi = F.interpolate(y[i:i+1, None, t:t+s, l:l+s].float(), size=(IMG_SIZE, IMG_SIZE), mode="nearest").long()[:, 0]
        k = random.randint(0, 3); xi, yi = torch.rot90(xi, k, (2, 3)), torch.rot90(yi, k, (1, 2))
        if random.random() < 0.5: xi, yi = xi.flip(3), yi.flip(2)
        g = 1 + (random.random() - 0.5) * 0.6; b = (random.random() - 0.5) * 50; sat = 1 + (random.random() - 0.5) * 0.6
        gray = xi.mean(1, keepdim=True); xi = ((xi - gray) * sat + gray) * g + b
        xs.append(xi.clamp(0, 255)); ys.append(yi)
    return torch.cat(xs).to(torch.uint8), torch.cat(ys)

def dice_loss(logits, y, cls=2):
    p = logits.softmax(1)[:, cls]; t = (y == cls).float()
    return 1 - (2 * (p * t).sum() + 1) / (p.sum() + t.sum() + 1)

def predict(m, X, bs=16):
    m.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            out.append(m(normalize(X[i:i+bs].to(device)))["out"].argmax(1).cpu())
    return torch.cat(out)

def evaluate(pred, Y, S_true):
    ious = []
    for c in range(len(CLASSES)):
        inter = ((pred == c) & (Y == c)).sum().item(); union = ((pred == c) | (Y == c)).sum().item()
        ious.append(inter / union if union else float("nan"))
    sev_pred = np.array([severity_of(p.numpy()) for p in pred])
    lv_t = np.array([level_of(v) for v in S_true]); lv_p = np.array([level_of(v) for v in sev_pred])
    r = float(np.corrcoef(sev_pred, S_true)[0, 1]) if np.std(sev_pred) > 0 and np.std(S_true) > 0 else float("nan")
    return {"iou_fondo": ious[0], "iou_hoja": ious[1], "iou_sintoma": ious[2], "miou": float(np.nanmean(ious)),
            "severity_mae_points": float(np.mean(np.abs(sev_pred - S_true)) * 100), "severity_pearson_r": r,
            "severity_level_accuracy": float(np.mean(lv_t == lv_p)),
            "severity_level_within_one": float(np.mean(np.abs(lv_t - lv_p) <= 1)), "n": int(len(Y))}

def show(name, m):
    print(f"  {name}: IoU fondo {m['iou_fondo']:.3f} | hoja {m['iou_hoja']:.3f} | síntoma {m['iou_sintoma']:.3f} | "
          f"mIoU {m['miou']:.3f} | severidad: error medio {m['severity_mae_points']:.2f} puntos, r = {m['severity_pearson_r']:.3f}, "
          f"nivel exacto {m['severity_level_accuracy']:.0%}, ±1 nivel {m['severity_level_within_one']:.0%} | n={m['n']}")

def ort_session(path):
    # Sin optimizaciones de grafo: misma aritmética que onnxruntime-web (WASM) en el teléfono.
    import onnxruntime as ort
    so = ort.SessionOptions(); so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    return ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])

def onnx_predict(path, X):
    sess = ort_session(path); out, t = [], 0.0
    for i in range(len(X)):
        x = normalize(X[i:i+1]).numpy(); t0 = time.time()
        out.append(sess.run(None, {"input": x})[0].argmax(1)[0]); t += time.time() - t0
    return torch.tensor(np.stack(out)), 1000 * t / max(1, len(X))

def plot_examples(out, m, X, Y, n=6):
    pred = predict(m, X[:n]); fig, axes = plt.subplots(3, n, figsize=(2.2 * n, 6.6))
    pal = np.array([[0, 0, 0], [0, 176, 0], [255, 0, 0]], np.uint8)
    for i in range(min(n, len(X))):
        axes[0, i].imshow(X[i].permute(1, 2, 0).numpy()); axes[1, i].imshow(pal[Y[i].numpy()]); axes[2, i].imshow(pal[pred[i].numpy()])
        for a in axes[:, i]: a.axis("off")
    axes[0, 0].set_title("foto", loc="left"); axes[1, 0].set_title("máscara real", loc="left"); axes[2, 0].set_title("predicción", loc="left")
    plt.tight_layout(); plt.savefig(f"{out}/ejemplos.png", dpi=110); plt.show(); plt.close(fig)

def run_model(name):
    print(f"\n==================== {name} ====================")
    out = f"{OUT_DIR}/{name}"; os.makedirs(out, exist_ok=True)
    m = build(name).to(device)
    Xtr, Ytr, _ = data["train"]; Xva, Yva, Sva = data["val"]
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    steps = EPOCHS * math.ceil(len(Xtr) / BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps)
    ce = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(device))
    best = -1.0
    for ep in range(EPOCHS):
        m.train(); perm = torch.randperm(len(Xtr)); tot, t0 = 0.0, time.time()
        for i in range(0, len(Xtr), BATCH):
            xb, yb = augment(Xtr[perm[i:i+BATCH]], Ytr[perm[i:i+BATCH]])
            xb, yb = normalize(xb.to(device)), yb.to(device)
            logits = m(xb)["out"]
            loss = ce(logits, yb) + dice_loss(logits, yb)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step(); tot += loss.item()
        ev = evaluate(predict(m, Xva), Yva, Sva)
        if (ep + 1) % 5 == 0 or ep == EPOCHS - 1 or SMOKE_TEST:
            print(f"Época {ep+1}/{EPOCHS} | pérdida {tot / math.ceil(len(Xtr) / BATCH):.3f} | val IoU síntoma {ev['iou_sintoma']:.3f} | "
                  f"mIoU {ev['miou']:.3f} | error severidad {ev['severity_mae_points']:.2f} pts | {time.time()-t0:.0f}s")
        score = ev["iou_sintoma"] if not math.isnan(ev["iou_sintoma"]) else ev["miou"]
        if score >= best:
            best = score; torch.save(m.state_dict(), f"{out}/best.pt")
    m.load_state_dict(torch.load(f"{out}/best.pt", map_location=device)); m.eval()

    Xte, Yte, Ste = data["test"]
    metrics = {"val": evaluate(predict(m, Xva), Yva, Sva), "test": evaluate(predict(m, Xte), Yte, Ste)}
    show("Validación", metrics["val"]); show(f"PRUEBA ({len(Yte)} hojas nunca usadas)", metrics["test"])
    plot_examples(out, m, Xte, Yte)

    # --- ONNX fp32 + int8 ---
    import onnx
    from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat
    exp = Export(m.to("cpu").eval())
    fp32, int8 = f"{out}/model_fp32.onnx", f"{out}/model_int8.onnx"
    kw = dict(input_names=["input"], output_names=["logits"], opset_version=17)
    try:
        torch.onnx.export(exp, torch.randn(1, 3, IMG_SIZE, IMG_SIZE), fp32, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(exp, torch.randn(1, 3, IMG_SIZE, IMG_SIZE), fp32, **kw)
    onnx.checker.check_model(onnx.load(fp32))
    class Calib(CalibrationDataReader):
        def __init__(self):
            n_tr = len(data["train"][0]); sel = random.Random(SEED).sample(range(n_tr), min(100, n_tr))
            Xc = F.interpolate(data["train"][0][sel].float(), size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
            self.it = iter([{"input": normalize(Xc[i:i+1]).numpy()} for i in range(len(Xc))])
        def get_next(self): return next(self.it, None)
    int8_ok = True
    try:
        pre = f"{out}/_pre.onnx"
        try:
            from onnxruntime.quantization.shape_inference import quant_pre_process
            quant_pre_process(fp32, pre)
        except Exception:
            shutil.copy(fp32, pre)
        quantize_static(pre, int8, Calib(), quant_format=QuantFormat.QDQ, per_channel=True,
                        activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
        os.remove(pre)
    except Exception as e:
        int8_ok = False; print("La cuantización falló:", e)

    sizes, onnx_m, preds = {}, {}, {}
    for tag, path in [("fp32", fp32), ("int8", int8)]:
        if tag == "int8" and not int8_ok: continue
        sizes[tag] = round(os.path.getsize(path) / 1e6, 2)
        preds[tag], ms = onnx_predict(path, Xte)
        onnx_m[tag] = evaluate(preds[tag], Yte, Ste); onnx_m[tag]["ms_per_image_cpu_unoptimized"] = round(ms, 1)
        metrics[f"onnx_{tag}_test"] = onnx_m[tag]
        show(f"ONNX {tag} prueba ({sizes[tag]} MB, {ms:.0f} ms/foto en CPU sin optimizar)", onnx_m[tag])
    stable = False
    if int8_ok:
        diff = abs(onnx_m["fp32"]["miou"] - onnx_m["int8"]["miou"])
        disagree = (preds["fp32"] != preds["int8"]).float().mean().item()
        stable = diff <= INT8_MAX_MIOU_DIFF and disagree <= INT8_MAX_PIXEL_DISAGREE
        metrics["int8_vs_fp32"] = {"miou_diff": diff, "pixel_disagreement": disagree, "stable": stable}
        print(f"  int8 vs fp32: diferencia de mIoU {diff:.3f} (máx. {INT8_MAX_MIOU_DIFF}), píxeles distintos {disagree:.1%} (máx. {INT8_MAX_PIXEL_DISAGREE:.0%})")
    if stable and sizes["int8"] <= MAX_MODEL_MB: rec = "int8"
    elif sizes["fp32"] <= MAX_MODEL_MB: rec = "fp32"
    else: rec = None
    if int8_ok and not stable: print("  La cuantización int8 cambia demasiado la segmentación: no se usa.")
    if rec is None: print(f"  {name} queda descartado: el fp32 pesa {sizes['fp32']} MB y el int8 no es confiable.")
    os.remove(f"{out}/best.pt")
    score = onnx_m[rec]["iou_sintoma"] if rec else -1.0
    return {"name": name, "out": out, "model": m, "metrics": metrics, "sizes": sizes, "recommended": rec,
            "score": score if not math.isnan(score) else onnx_m[rec]["miou"]}

results = []
for name in MODELS:
    try:
        results.append(run_model(name))
    except Exception as e:
        import traceback; traceback.print_exc(); print(f"\n*** {name} falló: {e}. Se sigue con el siguiente. ***")
    if device == "cuda": torch.cuda.empty_cache()

print("\nRESUMEN (prueba, aritmética del teléfono)")
print("modelo | archivo | MB | IoU hoja | IoU síntoma | error severidad (pts) | nivel exacto | ±1 nivel")
for r in results:
    t = r["metrics"].get(f"onnx_{r['recommended']}_test") if r["recommended"] else None
    if t: print(f"{r['name']} | {r['recommended']} | {r['sizes'][r['recommended']]} | {t['iou_hoja']:.3f} | {t['iou_sintoma']:.3f} | "
                f"{t['severity_mae_points']:.2f} | {t['severity_level_accuracy']:.0%} | {t['severity_level_within_one']:.0%}")
    else: print(f"{r['name']} | descartado")

## 4. Umbral de "¿hay una hoja en la foto?"

La app rechaza la foto si la fracción de píxeles de hoja + síntoma es menor que `min_leaf_fraction`. Se fija con datos:
hojas reales (BRACOL de prueba + JMuBEN de Kenia, otro estilo de foto) contra fotos que no son hojas (Food-101).

In [ ]:
def leaf_fraction(m, X):
    p = predict(m, X); return ((p == 1) | (p == 2)).float().mean((1, 2)).numpy()

def stream_images(name, split, n, key="image"):
    from datasets import load_dataset
    out = []
    for r in load_dataset(name, split=split, streaming=True):
        out.append(np.asarray(r[key].convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)))
        if len(out) >= n: break
    return torch.tensor(np.stack(out)).permute(0, 3, 1, 2).contiguous()

valid = [r for r in results if r["recommended"]]
assert valid, "Ningún modelo quedó por debajo de 10 MB con un archivo estable. Revisa los mensajes de arriba."
top = max(r["score"] for r in valid)
best = min([r for r in valid if r["score"] >= top - 0.01], key=lambda r: r["sizes"][r["recommended"]])
print("Elegido:", best["name"], best["recommended"])
bm = best["model"].to(device).eval()

frac_bracol = leaf_fraction(bm, data["test"][0])
gate = {"bracol_test_leaf_fraction_p05": float(np.percentile(frac_bracol, 5))}
frac_jm = frac_neg = None
if LEAF_GATE_SAMPLES:
    try:
        frac_jm = leaf_fraction(bm, stream_images("Project-AgML/arabica_coffee_leaf_disease_classification", "train", LEAF_GATE_SAMPLES))
        frac_neg = leaf_fraction(bm, stream_images("ethz/food101", "validation", LEAF_GATE_SAMPLES))
    except Exception as e:
        print("No se pudieron bajar las muestras para el umbral:", e)
if frac_jm is not None and frac_neg is not None:
    cands = np.linspace(0.02, 0.6, 59)
    # El umbral que mejor separa: maximiza (hojas aceptadas + no-hojas rechazadas), pero acepta al menos 97 % de las hojas.
    def score(t): return np.mean(frac_jm >= t) + np.mean(frac_neg < t)
    ok = [t for t in cands if np.mean(np.concatenate([frac_jm, frac_bracol]) >= t) >= 0.97]
    thr = float(max(ok, key=score)) if ok else float(np.percentile(np.concatenate([frac_jm, frac_bracol]), 2))
    gate.update({"jmuben_accepted": float(np.mean(frac_jm >= thr)), "bracol_test_accepted": float(np.mean(frac_bracol >= thr)),
                 "non_leaf_rejected": float(np.mean(frac_neg < thr)), "n_jmuben": int(len(frac_jm)), "n_non_leaf": int(len(frac_neg))})
    print(f"Umbral elegido: {thr:.2f} | acepta {gate['jmuben_accepted']:.0%} de hojas de JMuBEN y {gate['bracol_test_accepted']:.0%} "
          f"de BRACOL | rechaza {gate['non_leaf_rejected']:.0%} de fotos sin hoja")
else:
    thr = float(min(0.15, max(0.02, gate["bracol_test_leaf_fraction_p05"] * 0.5)))
    print(f"Umbral por defecto (sin muestras externas): {thr:.2f}")
gate["min_leaf_fraction"] = round(thr, 3)

## 5. Ficha del modelo y descarga

In [ ]:
def clean(x):
    if isinstance(x, float) and not math.isfinite(x): return None
    if isinstance(x, dict): return {k: clean(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [clean(v) for v in x]
    return x

final = f"{OUT_DIR}/final"; shutil.rmtree(final, ignore_errors=True); os.makedirs(final)
model_file = f"model_{best['recommended']}.onnx"
shutil.copy(f"{best['out']}/{model_file}", f"{final}/{model_file}")
if os.path.exists(f"{best['out']}/ejemplos.png"): shutil.copy(f"{best['out']}/ejemplos.png", final)

card = {
    "id": f"leafseg-v1-{best['name']}",
    "task": "segmentation",
    "architecture": best["name"],
    "classes": CLASSES,
    "input": {"name": "input", "shape": [1, 3, IMG_SIZE, IMG_SIZE], "layout": "NCHW", "color": "RGB",
              "resize": f"estirar a {IMG_SIZE}x{IMG_SIZE} (sin recorte)", "scale": "dividir entre 255", "mean": MEAN, "std": STD},
    "output": {"name": "logits", "shape": [1, len(CLASSES), IMG_SIZE, IMG_SIZE], "postprocess": "argmax por píxel"},
    "severity": {"formula": "pixeles_sintoma / (pixeles_hoja + pixeles_sintoma)",
                 "levels": [{"key": k, "max_fraction": v} for k, v in SEVERITY_LEVELS],
                 "levels_source": "BRACOL (Esgario et al.): sana < 0,1 %, muy baja 0,1–5 %, baja 5–10 %, alta 10–15 %, muy alta > 15 %"},
    "leaf_gate": gate,
    "files": {best["recommended"]: model_file},
    "recommended_file": model_file,
    "size_mb": {best["recommended"]: best["sizes"][best["recommended"]]},
    "metrics": best["metrics"],
    "candidates": [{"model": r["name"], "file": r["recommended"], "size_mb": r["sizes"], "score_iou_symptom": r["score"]} for r in results],
    "training": {"epochs": EPOCHS, "img_size": IMG_SIZE, "seed": SEED, "selection": "mejor IoU de síntoma en validación; int8 solo si es estable"},
    "training_data": [{"name": "BRACOL - segmentación (lara2018)", "country": "Brasil", "license": "MIT",
                       "source": "https://github.com/esgario/lara2018", "images": {k: len(v) for k, v in splits.items()}}],
    "known_limits": [
        "Entrenado con 400 hojas de Brasil fotografiadas sobre fondo claro: con fondos de campo la severidad puede ser menos precisa.",
        "La severidad mide área con síntoma, no la enfermedad: la clase la da el clasificador (arabica-v1).",
        "El umbral de '¿hay una hoja?' solo distingue hoja / no hoja; no reconoce otras plantas.",
    ],
}
if SMOKE_TEST: card["smoke_test"] = True
json.dump(clean(card), open(f"{final}/model_card.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False, allow_nan=False)
zip_path = shutil.make_archive("jani_leafseg", "zip", final)
print(json.dumps(clean({"metrics_test": best["metrics"]["test"], "leaf_gate": gate}), indent=2, ensure_ascii=False))
print("\nListo:", zip_path, "-> descomprimir dentro de app/public/models/leafseg-v1/")
try:
    from google.colab import files; files.download(zip_path)
except Exception:
    print("Descarga manual desde el panel de archivos:", zip_path)